# Spine 4 · Observed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dnakhoa/llm-engineering-playground/blob/main/spine/04-observed/observed.ipynb)

Every Case your Flagship Agent works now leaves an **OpenTelemetry trace**: an
agent span, a chat span per model call, a tool span per tool call, with tokens
and cost on them. You will read one, see the Checks catch a tool call that left
no span, and send a trace to an exporter the way you would send it to Jaeger or
any other backend.

Everything here runs **Offline**: replayed from reviewed recordings, no API key.
The lesson that goes with this notebook is `spine/04-observed/README.md`.

In [ ]:
# Setup. Locally this finds the course folder. On Colab it clones the course
# into the current folder once, and finds that clone again when you re-run this
# cell or restart the runtime.
import subprocess
import sys
from pathlib import Path

COURSE_URL = "https://github.com/dnakhoa/llm-engineering-playground.git"
CLONE = Path.cwd() / "llm-engineering-playground"


def find_course(start):
    for folder in [start, *start.parents, CLONE]:
        if (folder / "company").is_dir() and (folder / "llm").is_dir():
            return folder
    return None


COURSE = find_course(Path.cwd())
if COURSE is None:
    subprocess.run(["git", "clone", "--depth", "1", COURSE_URL, str(CLONE)], check=True)
    COURSE = CLONE
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("Course at", COURSE)

In [ ]:
# The OpenTelemetry SDK. Locally it comes with requirements.txt; on Colab this
# installs it once if the runtime does not have it.
try:
    import opentelemetry.sdk  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opentelemetry-sdk"], check=True)
print("OpenTelemetry SDK ready")

## 1. One Case, one trace

Run a Case as before. The Outcome now carries `trace`: every span the run
produced, as plain records. The tree below is built from each span's parent id.

In [ ]:
from company.runner import load_case, run_case

outcome = run_case(load_case("upgrade-to-pro"), "flagship.observed:run")


def print_tree(trace):
    children = {}
    for span in trace.spans:
        children.setdefault(span.parent_id, []).append(span)

    def walk(span, depth):
        print("{:<38} {:<8} {}".format("  " * depth + span.name, span.kind, span.status))
        for child in children.get(span.span_id, []):
            walk(child, depth + 1)

    for root in trace.roots:
        walk(root, 0)


print_tree(outcome.trace)

## 2. Tokens and cost on the spans

Each chat span carries its call's tokens, named as the GenAI conventions name
them, and its cost on `acme.cost_usd`: the conventions define tokens but not
money. The agent span carries the Case's totals. Both add up to the Outcome.

In [ ]:
for span in outcome.trace.chat_spans:
    a = span.attributes
    print("{:<12} in={:<5} out={:<4} cost=${:.6f}".format(
        a["gen_ai.response.id"], a["gen_ai.usage.input_tokens"],
        a["gen_ai.usage.output_tokens"], a["acme.cost_usd"]))

(agent_span,) = outcome.trace.agent_spans
print()
print("agent span: in={} out={} cost=${:.6f}".format(
    agent_span.attributes["gen_ai.usage.input_tokens"],
    agent_span.attributes["gen_ai.usage.output_tokens"],
    agent_span.attributes["acme.cost_usd"]))
print("Outcome:    in={} out={} cost=${:.6f}".format(
    outcome.usage.input_tokens, outcome.usage.output_tokens, outcome.cost_usd))

## 3. The pinned convention version

The GenAI conventions are still in Development, so their names can change
between releases. The course pins one, in `company/tracing.py`, and every span
says which in its schema URL.

In [ ]:
from company.tracing import SCHEMA_URL, SEMCONV_VERSION

print("OpenTelemetry semantic conventions", SEMCONV_VERSION)
print({span.schema_url for span in outcome.trace.spans} == {SCHEMA_URL}, SCHEMA_URL)

## 4. The agent says who it is

The environment traces every `env.complete` and `env.act`, so the Spine 2 agent
is traced without a change. What it cannot know is the agent's name. The Spine 4
agent adds it, and the Spine 2 agent gets a warning on that one Check: `WARN`,
which fails nothing, because the conventions set the name only "if provided by
the application".

In [ ]:
import inspect

import flagship.observed
from checks.spine_4_observed import CHECKS

print(inspect.getsource(flagship.observed.run))


def grade(outcome):
    for check in CHECKS:
        result = check(outcome)
        status = "FAIL" if not result.passed else "WARN" if result.warning else "PASS"
        print(status, "-", result.name, "-", result.detail)


grade(run_case(load_case("upgrade-to-pro"), "flagship.knowledge:run"))

## 5. A tool call with no span

This agent answers Knowledge Base searches itself, with the same retriever and
the same result, instead of through `env.act`. The Backend ends right and the
replay matches. But no span says the search happened, and the Check says so.

In [ ]:
from company.knowledge import SEARCH_TOOL, run_search
from company.runner import StepLimitReached
from flagship.loop import SYSTEM
from llm.types import Message


def searches_by_itself(customer_turn, env):
    env.describe_agent("acme-support")
    messages = env.memory.setdefault("transcript", [])
    messages.append(Message.user(customer_turn))
    for _ in range(env.step_limit):
        response = env.complete(messages, tools=env.tools, system=SYSTEM)
        messages.append(Message.assistant(response.text or None, response.tool_calls))
        if not response.tool_calls:
            return response.text
        messages.append(Message.tool([
            run_search(env.knowledge, call) if call.name == SEARCH_TOOL.name else env.act(call)
            for call in response.tool_calls
        ]))
    raise StepLimitReached(env.step_limit)


untraced = run_case(load_case("downgrade-with-prorated-refund"), searches_by_itself)
print("resolved:", untraced.resolved)
grade(untraced)

## 6. Errors are on the spans

An Action the Case does not offer is refused. The refusal is an `execute_tool`
span with status `ERROR`, which is what you filter on in a tracing backend.
This agent never asks the model, so it needs no recording.

In [ ]:
from llm.types import ToolCall


def refunds_unasked(customer_turn, env):
    env.describe_agent("careless-agent")
    env.act(ToolCall("x1", "issue_refund", {
        "account_id": "acct_1001", "invoice_id": "inv_1001", "amount_usd": 5.00}))
    return "Refunded!"


refused = run_case(load_case("upgrade-to-pro"), refunds_unasked)
(refused_span,) = refused.trace.tool_spans
print(refused_span.name, refused_span.status, refused_span.attributes["error.type"])

## 7. Export to any OpenTelemetry backend

Pass span exporters to `run_case`, and the same spans go to them as they end.
An in-memory exporter stands in for a backend here. For Jaeger, Tempo,
Honeycomb, Langfuse or your cloud's tracing, pass an `OTLPSpanExporter`
instead (`pip install opentelemetry-exporter-otlp-proto-http`); the lesson
shows how.

In [ ]:
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

backend = InMemorySpanExporter()
for case_id in ["upgrade-to-pro", "annual-refund-outside-window"]:
    run_case(load_case(case_id), "flagship.observed:run", exporters=[backend])

exported = backend.get_finished_spans()
print(len(exported), "spans in", len({s.context.trace_id for s in exported}), "traces")

# With a real backend:
# from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
# exporter = OTLPSpanExporter(endpoint="http://localhost:4318/v1/traces")
# run_case(load_case("upgrade-to-pro"), "flagship.observed:run", exporters=[exporter])

## 8. The Checks, cumulatively

Running module 4 runs every earlier module's Checks too.

In [ ]:
from checks.cli import run_checks

report = run_checks(through=4, agent="flagship.observed:run")

## Your turn

* Name your own agent with `env.describe_agent`, and run
  `python -m checks --modules 4 --agent path/to/my_agent.py:run`.
* Start a span of your own with `env.tracer.start_as_current_span("plan")`
  inside your agent, and find it under the agent span with `print_tree`.
* Run Jaeger locally, export every Case to it, and find the refused refund
  from section 6 by its error status.